## 1st Day CODE Notes

1. We have built AlphaDesk’s research intake workflow.
    - It accepts a company 'ticker' and 'research question', validates the input, and creates a structured task for the future Fundamentals Agent.

    - At this stage, the application prepares work. It does not yet retrieve financial documents, call an LLM, or produce an investment memo.

2. Code Structure
    - schemas.py : Define the data agreements ( What the data should look like)
    - nodes/validate_request.py : Normalize and validate user input ( What each workflow step does)
    - nodes/prepare_task.py : Create a task from validated input(What each workflow step does)
    - graphs/research.py : Connect the nodes into an executable workflow ( Which step runs next)
    - cli.py : Accept terminal arguments, execute the graph, display results (How a terminal user interacts with the application)
    - main.py : Enable execution through python -m alphadesk

## FLOW LIKE THIS:

3. schemas.py :
    - ResearchRequest : What input we accept and validate
    - ResearchTask : What one prepared work item looks like
    - ResearchState : What information travels through the graph

4. validate_request.py : Normalize and validate input

5. prepare_task.py : Create the work item

6. graphs/research.py : Assemble the workflow

7. cli.py : It is the terminal-facing entry to our application. Its responsibilities are:
    - Read arguments.
    - Construct the workflow
    - Submit input
    - Handle expected input errors.
    - Display the result
    - Understanding the command : uv run python -m alphadesk --ticker AAPL --question "How healthy is Apple?"
    - argparse : command-line parser.


# PART 2

- WHAT we will do now:
    - We’ll build the first India-focused evidence pipeline: take an official annual-report PDF ( Infosys FY2024–25 ), preserve the original, and extract text with page references.

    - FLOW : Official PDF + source details -> Validate -> fingerptint -> extarct pages -> save evidence

 - Outputs and purpose like this below :
    - source.pdf : Preserved original document
    - pages.jsonl : Extracted text , one record per PDF page
    - manifest.json : Source details,fingerprints,extraction version,summary

- Our existing research-intake graph remains unchanged. We are building its evidence supply first.

- Later:
Research question → document-search tool → relevant pages → Fundamentals Agent → cited answer

## MUST KNOW CONCEPTS:
- Annual Report : Company's yearly business and financial report
- Financial year: reporting year
- Standalone/consolidated: Company-only account/accounts including subsidiaries
- Provenance: where info came from and how we processed it
- Content hash: fingerptint of file's bytes
- JSONL: text file containing one JSON object per line
- OCR: Recognizing text in scanned images
- uv add pymupdf : for PyMuPDF’s text extraction with sorting

## CODE FLOW:
- Contracts : define valid metadata and page records
- Services : Perform ingestion independently
- CLI : Read Arguments and call the service


## FLOW SO FAR
* ``documents.py defines what data should look like and ingestion.py defines how we create and save data ( read pdf , Extract text , validate page records , write output files )``

1. Think of the first as receiving a research question, and the second as preparing the library of documents needed to answer it.
2. How our previous feature works:
    - User enters ticker + question
            - → ResearchRequest validates the input
            - → Graph prepares a ResearchTask
            - → Task says “pending_research”
3. It stops there because we haven’t supplied searchable evidence yet.
4. Where documents.py comes into the new feature:
    - We now need to process an annual report. Before writing that processing logic, we define what its input and output should look like.
    - documents.py contains two classes:
        - AnnualReportMetadata : The report’s identity and source details : Record which company, financial year, and source the document belongs to
        - PageEvidence: Text extracted from one PDF page : Keep the text connected to a document and page for later citations.

5. The ingestion flow is:
    - PDF + metadata JSON :
        -  → AnnualReportMetadata validates the supplied metadata
        - Ingestion service opens the PDF
        - Each page becomes a validated PageEvidence record
        - Save original PDF, page records, and manifest

6. For Example : Page Record :
    {
    "document_id": "<document fingerprint>",
    "evidence_id": "<document fingerprint>:page:25",
    "pdf_page_number": 25,
    "pdf_page_label": None,
    "text": "Text extracted from this page...",
    "extraction_status": "text_extracted",
     }

     - Without those identifiers, we might have extracted text but lose the ability to answer: “Which report and page support this statement?”

7. The eventual connection will be:
    - Ingestion prepares page-linked evidence
                    ↓
          Searchable document corpus
                    ↑
Research task → Search tool → Relevant evidence → Agent’s cited answer

8. Nw crtceate the ingestion Service :
    - Checks File Size
    - Computes the document fingerprint
        - fingerprint is a vlaue calculated from files content using algo such as SHA-256
        - indetical files content produces same fingerprint, even if filanemae changes
        - in AlphaDesk, it helps identify duplicate reports
    - Opens the PDF , calculates its fingerprints , and extracts text page by page
    - Extracts one record per page
    - Validates each page record and saves the original PDF , extracted text, and manifest
    - writes everything into temporary directory
    - Publishes the completed Directory after success



## So What Have We done Till now ?

1. We have built two foundations: receiving a research request and preparing company documents as evidence. They are not connected yet.
2. AlphaDesk will eventually follow this flow:
    - Research request Query → retrieve company evidence → specialist agents analyse it → verify claims → draft memo → human approval

3. What we have worked on so far supports the first two parts.
A). First feature: receive a research request:
    - Graph : Start → validate request → prepare task → end
    - AlphaDesk understands the request and creates a consistent task.
    - What it does not achieve yet: researching the company. The task remains pending_research.

B).Second feature: prepare document evidence :
    - PDF + metadata → validate metadata → fingerprint PDF → extract pages → save evidence
    - Annual Report PDF : Original company document
    - Metadata JSON : Company, symbol, financial year, and source URL
    - contracts/documents.py : Defines valid report metadata and page-evidence records
    - ingestion/ingest_report.py : Reads ingestion arguments and calls the service ( CLI )
    - services/pdf_ingestion.py : Processes the PDF and saves the results

    - STEPS INSIDE PDF INGESTION :
        1. Check the input: confirm the PDF exists and meets the size limit.
        2. Calculate its fingerprint: SHA-256 identifies the exact source bytes.
        3. Prepare temporary storage: avoid exposing unfinished output as completed.
        4. Preserve the PDF: save the original as source.pdf.
        5. Open and inspect it: reject unsupported encrypted documents or excessive page counts.
        6. Extract text page by page: retain physical page numbers and available labels.
        7. Validate each page record: construct a PageEvidence object.
        8. Write evidence: save one page record per line in pages.jsonl.
        9. Write provenance: save source and processing details in manifest.json.
        10. Publish completed output: move the staged directory to its final location.

    - The intended outputs are:
     1. source.pdf : The exact original used
     2. pages.jsonl : Text connected to individual pages
     3. manifest.json :Source identity, hash, parser version, and extraction summary


# We will now Build Chunking
- Turn each extracted PDF page into smaller chunks that a search tool can retrieve
    - PDF Ingestion ----> Preserve the report and extract page text ---> Already Done
    - Chunking ----> Create smaller passages with source references ----> Building now
    - Indexing ----> Store searchable representation of those passages ----> Next
    - Search Tool ----> Return passages relevant to a quesation ----> Later
    - Fundamentals Agent ----> Use retrieved evidence to answer with citations ----> Later 

- New Concepts :
    - chunksize : maximum passage length
    - overlap : text repetae dbetween adjacent chunks so context is less likely to be cut
    - character offsets : positions identifying exactly where the chunk came from within extracted page text
    - chunk id : stable identifier to one passage
    - chunjk manifest : processing details how chunk was processed

- chunk alphadesk/contracts/chunk.py ---> chunks.py
    - this file describes one searchable passage.
    - Each chunk stores :
        - its identity and original document identity
        - company identifier
        - physical pdf page number
        - start and end positions with that page's extracted text
        - passage text


# Embeddings

- FLOW :  PDF -> pages -> chunks -> embeddings -> similarity search -> Qdrant -> Agent
- We'll generate Embeddings and check whether question retrievws relevant chunks. ( No Agent yet )
- We will use  ``BAAI/bge-small-en-v1.5`` Embedding Model
- ``QDRANT`` : Embedding Models the Vectors and QDRANT stores , indexes and searches the relevant passages to or Query.

# FLOW DIAGRAM SO FAR
## Research Intake - 

![mermaid-diagram (1).png](<attachment:mermaid-diagram (1).png>)


# 2. Document preparation → chunks → embeddings

## ![mermaid-diagram (2).png](<attachment:mermaid-diagram (2).png>)

# 3. The embedding-search check we supplied

## ![mermaid-diagram (3).png](<attachment:mermaid-diagram (3).png>)

# QDRANT
- We will now put saved Embeddings into Qdrant ( Enterprise Vector Embeddings ). This will make the passages available with Application through Database,
- PDF -> pages ->  chunks -> Embeddings -> Qdrant -> Search Tool -> AI Answer ( Agent )

### ``CONCEPTS``

1. ``Collection``:         Named group of Searchable records in Qdrant
2. ``Point``:              One stored Record : ID , vector , associated information
3. ``Payload`` :           PAssage text , metadata and vector
4. ``Upsert`` :            Insert a point( Record ) if absent ; replace it if teh same ID already exists.
5. ``Cosine Similarity``:  Way to compare direction of vector for similarity
6. ``Docker Volume`` :     Storage kept outside the database container's writable layer

- FOR EVERY CHUNK , we will store :
    ``POINT``:
        -``ID``
        -``Vector``
        -``Payload``
            -``chunk_id``
            -``document_id``
            -``pdf_page_number``
            -``text``
            -``embeddingmodel``

# START QDRANT

## compose.qdrant.yaml

- After writing that file
- docker compose -f compose.qdrant.yaml up -d
- curl --fail http://127.0.0.1:6333
- uv add qdrant-client

# Now Build Indexing Service - alphadesk/services/vector_store.py

1. Read teh saved Embedding records
2. validate each chunks text and Citation Fields
3. check the model, vector size , numerical values
4. Convert each record to Qdrant Point
5. Connect to Qdrant
6. Create Collection if needed
7. Upload points in small batches
8. Return the number submitted



# RETRIEVAL

- We have indexed passages.  Now we will retrieve passages using a query/question
- Question -> Embedding model -> Question Vector -> Qdrant -> Matching Passages
- ``QUERY``: question we want to serach answer for from documents
- ``Query Embedding`` : question converted into vectors using same model as used for passages
- ``top_k`` : how many mathcing passages to return
- ``Similarity Score``: how closely a passage vector matches the question vector
- create retrieval.py:
    1. Loads our embedding model.
    2. Converts the question into a vector.
    3. Searches the existing Qdrant collection.
    4. Returns the matching points, including their text and page references.
    We reuse the constants from vector_store.py so indexing and searching use the same settings.

# TO DO

| Capability | Status | Purpose |
|---|---|---|
| **Vector search** | Implemented with embeddings + Qdrant | Find passages similar in meaning to your question |
| **Hybrid search** | Planned | Combine vector search with keyword search for exact financial terms, names, and identifiers |
| **Reranking** | Planned | Reorder retrieved passages using a model that evaluates each passage against the question |
| **Metadata filters** | Planned | Restrict retrieval to a company, document, or reporting year |
| **Caching** | Planned | Reuse selected computations or results to reduce latency and cost; refresh when underlying data changes |